In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import pandas as pd
import os

In [ ]:
# デスクトップパス
# 環境変数から取得。OS判定が必要。
if os.name == 'nt':
    home = os.getenv('USERPROFILE')
else:
    home = os.getenv('HOME')

desktop_dir = os.path.join(home, 'Desktop')
desktop_dir = desktop_dir.replace("\\", "/")

# メイン
pathMainFolder = desktop_dir + '/Python/business_management' + '/'

# プロファイル
path_profile = desktop_dir + "/Python/data/selenium_profile"

# ------------------------------------------
# output Excel file path/name
# ------------------------------------------
finleName = 'email_address'
suffix = ''

output_excel_path = pathMainFolder + finleName + suffix + '.xlsx'

In [ ]:
# 読込関数
def read_text_file(path):
    for enc in ["utf-8-sig", "cp932"]:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read().splitlines()
        except UnicodeDecodeError:
            pass
    raise UnicodeDecodeError("The character code could not be determined.")

# 古いメモ帳（～2019頃まで）：既定の保存形式は、ANSI（cp932 / Shift_JIS）
# 新しいメモ帳（Windows 10以降）：既定の保存形式は、UTF-8（BOM付き）

IDリストの読み込み

In [ ]:
textName = 'id.txt'
pathTextID = pathMainFolder + textName

listID = read_text_file(pathTextID)

cntID = len(listID)

ChromeDriver

In [ ]:
service = Service(ChromeDriverManager().install())

time.sleep(3)

def create_driver(path_profile, service):
    options = Options()
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})

    # 通常Chromeプロファイルを使う
    pathProfile = path_profile
    strArgument = "--user-data-dir=" + pathProfile
    options.add_argument(strArgument)
    
    # ChromeDriverを自動的にダウンロード・設定
    driver = webdriver.Chrome(service=service, options=options)

    time.sleep(1)
    return driver

関数定義

In [ ]:
# ------------------------------------------
# Goボタンが活性化されたらボタンを押下
# ------------------------------------------
def click_go_when_ready(driver):
    WebDriverWait(driver, 10).until(
        lambda d: d.execute_script("""
            function findGo(root) {
              const walker = document.createTreeWalker(
                root,
                NodeFilter.SHOW_ELEMENT,
                null,
                false
              );

              let node = walker.currentNode;

              while (node) {
                if (
                  node.tagName === 'DIV' &&
                  node.classList.contains('scl-clickable') &&
                  node.textContent &&
                  node.textContent.trim() === 'Go'
                ) {
                  // 表示されているか
                  const style = window.getComputedStyle(node);
                  const visible = style.display !== 'none' &&
                                  style.visibility !== 'hidden' &&
                                  node.offsetParent !== null;

                  // disabledクラスがないか（サイト仕様に応じて）
                  const enabled = !node.classList.contains('disabled');

                  if (visible && enabled) {
                    node.click();
                    return true;
                  }
                }

                if (node.shadowRoot) {
                  if (findGo(node.shadowRoot)) {
                    return true;
                  }
                }

                node = walker.nextNode();
              }

              return false;
            }

            return findGo(document);
        """)
    )


# ------------------------------------------
# タブ数が増えたか確認（主にGoボタン押下後）
# ------------------------------------------
def wait_for_new_tab(driver, wait, before_handles):
    try:
        wait.until(lambda driver: len(driver.window_handles) > before_handles)
        return True
    # except TimeoutException:
    except:
        return False

ループ処理

In [ ]:
# IDの検索ページ
url_input = 'https://～～～input'

# 取得したいデータの格納先
url_containing_the_values_we_need = 'https://～～～containing_value'

# 取得したデータを入れるリストの定義
my_list = []

wait = WebDriverWait(driver, 10)

for idx, id in enumerate(listID):

    # ------------------------------------------
    # 500回ごとに再起動（処理の安定化のため）
    # ------------------------------------------
    if idx != 0 and idx % 500 == 0:
        driver.quit()
        time.sleep(10)
        driver = create_driver(path_profile, service)
        time.sleep(1)
        wait = WebDriverWait(driver, 10)
        time.sleep(1)

    # ------------------------------------------
    # IDの検索ページにアクセス
    # ------------------------------------------
    driver.get(url_input)

    time.sleep(2)

    WebDriverWait(driver, 10).until(lambda d: d.execute_script("return document.readyState") == "complete")

    # ------------------------------------------
    # ID 検索ボックスへの値の入力
    # ------------------------------------------
    # ① Shadow host を取得
    host = wait.until(lambda d: d.find_element(By.ID, "input-box"))
    
    # ② shadowRoot を取得
    shadow_root = driver.execute_script("return arguments[0].shadowRoot", host)
    
    # ③ Shadow DOM 内の input を取得
    input_box = shadow_root.find_element(By.CSS_SELECTOR, "input[type='search']")
    
    input_box.send_keys(id)

    time.sleep(0.1)
    # ------------------------------------------
    # Input ボタンの押下
    # ------------------------------------------
    # Shadow DOM を再帰的に全部探索する（JSで全 shadowRoot を総当たり）
    driver.execute_script("""
    function findAndClickInput(root) {
      // 通常DOM + shadowRoot 両方を探索
      const treeWalker = document.createTreeWalker(
        root,
        NodeFilter.SHOW_ELEMENT,
        null,
        false
      );
    
      let node = treeWalker.currentNode;
    
      while (node) {
        // button 本体を判定
        if (
          node.tagName === 'BUTTON' &&
          node.textContent &&
          node.textContent.trim() === 'Input'
        ) {
          node.click();
          return true;
        }
    
        // shadowRoot があれば潜る
        if (node.shadowRoot) {
          if (findAndClickInput(node.shadowRoot)) {
            return true;
          }
        }
    
        node = treeWalker.nextNode();
      }
      return false;
    }
    
    if (!findAndClickInput(document)) {
      throw 'Input button not found anywhere';
    }
    """)
    
    time.sleep(2)
    # ------------------------------------------
    # Go ボタンの押下
    # ------------------------------------------
    # Goボタン押下後の待機のためにあらかじめタブ数を取得
    WindowCountBeforeGo = len(driver.window_handles)

    click_go_when_ready(driver)

    time.sleep(1)
    # ------------------------------------------
    # Goボタン押下後、タブ数が増えるまで待つ
    # ------------------------------------------
    tab_opened = wait_for_new_tab(driver, wait, WindowCountBeforeGo)

    if not tab_opened:
        # 新タブが開かなかった場合の処理
        my_list.append("ERROR: tab not opened")
        # 念のためタブ数を確認し、増えていれば閉じる
        if len(driver.window_handles) > WindowCountBeforeGo:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
        continue  # 次のIDへ
    
    time.sleep(0.25)
    # ------------------------------------------
    # 必要とするデータの格納先での転記処理
    # ------------------------------------------
    # 現在のウィンドウハンドル（タブのリスト）を取得
    handles = driver.window_handles

    time.sleep(0.25)
    
    # 最後のタブに切り替える
    driver.switch_to.window(handles[-1])

    time.sleep(0.25)

    # 転記処理開始 ------------------------------------------
    try:
        # 転記処理 (1)
        driver.get(url_containing_the_values_we_need)
        time.sleep(0.3)
    
        WebDriverWait(driver, 10).until(lambda d: d.execute_script("return document.readyState") == "complete")
    
        # 転記処理 (2)
        email_elem = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div.email")
            )
        )
        
        email_elem = email_elem.text.split("\n")[0]
    
        my_list.append(email_elem)
    
        time.sleep(0.25)

    except Exception as e:
        my_list.append("")
    # 転記処理終了 ------------------------------------------

    finally:
        try:
            # 最後のタブを閉じる
            driver.close()
            
            # 最初のタブに戻る
            driver.switch_to.window(driver.window_handles[0])
        
            time.sleep(0.25)
        except Exception:
            pass  # 万が一タブ操作自体が失敗してもループは継続させる

In [ ]:
driver.quit()

DataFrame

In [ ]:
# listをpandasのseriesに変換
series_my_list = pd.Series(my_list)
series_listID  = pd.Series(listID)

# 結合
df = pd.concat([series_listID, series_my_list], axis=1)

# 列名の変更
df.columns = ['id', 'email_address']

In [ ]:
df_dump = df
df_dump.to_excel(output_excel_path, sheet_name = 'Sheet1', index=False)